# RAG Pipeline — Step by Step Walkthrough

Run each cell one at a time. Each step shows what's happening inside the pipeline.

```
Step 1 → Load Data
Step 2 → Preprocess
Step 3 → Chunking
Step 4 → Embedding
Step 5 → Vector DB (store)
Step 6 → Query Classification
Step 7 → Retrieval
Step 8 → Reranking
Step 9 → Answer Generation  ← needs Ollama running
Step 10 → Evaluation        ← needs Ollama running
```

> Steps 1-8 work without Ollama. Steps 9-10 need `ollama serve` running with `llama3.1:8b`.

In [3]:
# ── Setup: make src/ importable from this notebook ──────────────────────────
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
print('Python path set. Ready.')

Python path set. Ready.


---
## Step 1 — Load Data
Load one row from the finance dataset (finqa) in RAGBench.
Each row has: `question`, `documents` (list of context strings), `answer`.

In [5]:
from datasets import load_dataset

print('Loading finqa from RAGBench (test split)...')
ds = load_dataset('rungalileo/ragbench', 'finqa', split='test')

# Pick one example row
row = ds[0]

print(f'Total rows in dataset: {len(ds)}')
print(f'\nColumns available: {ds.column_names}')
print(f'\n{"-"*50}')
print(f'QUESTION:\n{row["question"]}')
print(f'\nGOLD ANSWER:\n{row["response"]}')
print(f'\nNumber of context documents in this row: {len(row["documents"])}')
print(f'\nFirst context document (first 300 chars):')
print(row['documents'][0][:300])

Loading finqa from RAGBench (test split)...
Total rows in dataset: 2294

Columns available: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']

--------------------------------------------------
QUESTION:
what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?

GOLD ANSWER:
The rate of return in Cadence Design Systems Inc. from 2010 to 2011 is 37.9%. This is calculated by taking the value on 1/1/2011 (137.90) and subt

---
## Step 2 — Preprocess
Clean the raw document text before chunking (removes excess newlines).

In [6]:
from ingestion.preprocess_text import preprocess_documents

raw_doc = row['documents'][0]
cleaned_doc = preprocess_documents(raw_doc)

print(f'Original length : {len(raw_doc)} characters')
print(f'Cleaned length  : {len(cleaned_doc)} characters')
print(f'\nCleaned text (first 400 chars):')
print(cleaned_doc[:400])

Original length : 1092 characters
Cleaned length  : 1092 characters

Cleaned text (first 400 chars):
stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinves


---
## Step 3 — Chunking
Split the cleaned document into smaller overlapping chunks.

**Why?** LLMs and vector DBs work best with smaller pieces of text (~500-1000 chars), not entire documents.

In [7]:
from chunking.text_chunking import chunk_documents

chunks = chunk_documents(cleaned_doc, chunk_size=1000, chunk_overlap=100)

print(f'Document split into {len(chunks)} chunks')
print(f'Chunk size setting : 1000 chars | Overlap : 100 chars')
print()

for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk[:250])
    if len(chunk) > 250:
        print('...')
    print()

Document split into 2 chunks
Chunk size setting : 1000 chars | Overlap : 100 chars

--- Chunk 1 (999 chars) ---
stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . 
...

--- Chunk 2 (190 chars) ---
12/31/09 in index , including reinvestment of dividends . indexes calculated on month-end basis . copyright a9 2014 s&p , a division of the mcgraw-hill companies inc . all rights reserved. .



---
## Step 4 — Embedding
Convert each chunk into a vector (a list of numbers).

**Why?** Vectors let us do math-based similarity search — similar texts have similar vectors.

In [8]:
from embeddings.embedder import get_embedder
import numpy as np

print('Loading embedding model (BAAI/bge-small-en-v1.5)...')
embedder = get_embedder()

# Embed the first chunk
sample_chunk = chunks[0]
vector = embedder.encode(sample_chunk)

print(f'\nChunk text (first 100 chars): {sample_chunk[:100]}...')
print(f'\nEmbedding vector shape  : {vector.shape}')   # e.g. (384,)
print(f'Vector dimension        : {len(vector)}')
print(f'First 10 values         : {np.round(vector[:10], 4)}')
print(f'Min value               : {vector.min():.4f}')
print(f'Max value               : {vector.max():.4f}')

# Show similarity between two chunks
if len(chunks) >= 2:
    vec2 = embedder.encode(chunks[1])
    similarity = np.dot(vector, vec2) / (np.linalg.norm(vector) * np.linalg.norm(vec2))
    print(f'\nCosine similarity between Chunk 1 and Chunk 2: {similarity:.4f}')
    print('(1.0 = identical, 0.0 = unrelated)')

Loading embedding model (BAAI/bge-small-en-v1.5)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\az845\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\az845\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Chunk text (first 100 chars): stockholder return performance graph the following graph compares the cumulative 5-year total stockh...

Embedding vector shape  : (384,)
Vector dimension        : 384
First 10 values         : [-0.0513 -0.0231 -0.0122 -0.0053  0.0221 -0.0049 -0.0257  0.0646  0.0223
  0.012 ]
Min value               : -0.3098
Max value               : 0.3418

Cosine similarity between Chunk 1 and Chunk 2: 0.7668
(1.0 = identical, 0.0 = unrelated)


---
## Step 5 — Vector DB (Store)
Store a small set of chunks into ChromaDB so we can search them.

We use an **in-memory** ChromaDB here (not persistent) so we can test quickly without polluting the real DB.

In [ ]:
import chromadb
from chromadb import EmbeddingFunction, Embeddings

# Wrap the already-loaded embedder from Step 4 — no model re-download
class ReuseEmbedder(EmbeddingFunction):
    def __call__(self, input):
        return embedder.encode(input).tolist()

client = chromadb.EphemeralClient()
collection = client.get_or_create_collection(
    name='finance_demo',
    embedding_function=ReuseEmbedder()
)

# Use chunks already created in Step 3 (from 1 document only — keeps it fast)
ids   = [f'chunk_{i}' for i in range(len(chunks))]
texts = chunks

collection.add(documents=texts, ids=ids)

print(f'Stored {collection.count()} chunks in ChromaDB')
print(f'\nChunk IDs stored:')
for cid in ids:
    print(f'  {cid}')


---
## Step 6 — Query Classification
Before retrieving, the pipeline classifies the query to decide:
- Does this question need RAG (retrieval)?
- Or can the LLM answer it directly from its own knowledge?

> **Needs Ollama running.** Skip to Step 7 if you don't have it yet.

In [ ]:
# ── Uncomment this cell when Ollama is running ────────────────────────────────

# from classification.query_classifier import build_classifier
# from classification.query_strategy import determine_strategy
# from llm.model import llm

# query = row['question']
# print(f'Query: {query}')

# classifier = build_classifier(llm=llm)
# classification = classifier.invoke({'query': query})

# print(f'\nClassification result:')
# print(f'  Domain             : {classification["domain"]}')
# print(f'  Dataset            : {classification["dataset"]}')
# print(f'  Retrieval required : {classification["retrieval_required"]}')
# print(f'  Reasoning          : {classification["reasoning"]}')

# strategy = determine_strategy(classification)
# print(f'\nStrategy chosen: {strategy}')

print('Cell is commented out — uncomment when Ollama is running.')

---
## Step 7 — Retrieval
Search the vector DB for chunks most similar to the query.

ChromaDB computes cosine similarity between the query embedding and all stored chunk embeddings.

In [ ]:
query = row['question']
TOP_K = 5

print(f'Query: {query}')
print(f'Retrieving top {TOP_K} chunks...\n')

results = collection.query(
    query_texts=[query],
    n_results=min(TOP_K, collection.count())
)

retrieved_docs  = results['documents'][0]
retrieved_ids   = results['ids'][0]
retrieved_dists = results['distances'][0]

print(f'Retrieved {len(retrieved_docs)} chunks:\n')
for i, (doc_id, dist, doc) in enumerate(zip(retrieved_ids, retrieved_dists, retrieved_docs)):
    print(f'[{i+1}] ID: {doc_id} | Distance: {dist:.4f}')
    print(f'    {doc[:200]}...')
    print()

---
## Step 8 — Reranking
The initial retrieval uses vector similarity (fast but approximate).
Reranking uses a **cross-encoder** model that reads the query + each doc together for a more accurate relevance score.

The cross-encoder is slower but much more precise — it's a second-pass filter.

In [ ]:
from reranking.reranker import rerank

TOP_K_RERANK = 3

print(f'Reranking {len(retrieved_docs)} retrieved chunks → keeping top {TOP_K_RERANK}\n')

reranked_docs = rerank(query, results)[:TOP_K_RERANK]

print(f'\nTop {TOP_K_RERANK} docs after reranking:')
for i, doc in enumerate(reranked_docs):
    print(f'\n[{i+1}] {doc[:300]}...')

---
## Step 9 — Answer Generation
Send the query + top reranked chunks to the LLM to generate an answer.

> **Needs Ollama running**: `ollama serve` + `ollama pull llama3.1:8b`

In [ ]:
# ── Uncomment when Ollama is running ─────────────────────────────────────────

# from llm.answer_generation import generate_answer
# from prompts.answer_prompt import ANSWER_PROMPT

# print(f'Query: {query}\n')
# print('Prompt sent to LLM (preview):')
# print(ANSWER_PROMPT.format(query=query, context=reranked_docs[:2])[:600])
# print('...')
# print('\nGenerating answer...')

# answer = generate_answer(query, reranked_docs)
# print(f'\nGENERATED ANSWER:\n{answer}')
# print(f'\nGOLD ANSWER:\n{row["answer"]}')

print('Cell is commented out — uncomment when Ollama is running.')

---
## Step 10 — Evaluation
Score the generated answer using an LLM-as-judge that computes the 4 RAGBench metrics.

| Metric | What it measures |
|---|---|
| Context Relevance | Were the retrieved chunks relevant to the question? |
| Context Utilization | Did the LLM actually use what was retrieved? |
| Completeness | Did the answer cover all relevant facts? |
| Adherence | Is every claim in the answer backed by the context? |

> **Needs Ollama running**

In [ ]:
# ── Uncomment when Ollama is running ─────────────────────────────────────────

# from evaluation.evaluation_metrics import evaluate_rag_response, calculate_metrics

# eval_result = evaluate_rag_response(
#     question=query,
#     contexts=reranked_docs,
#     answer=answer
# )

# metrics = calculate_metrics(eval_result)

# print('RAG Evaluation Results:')
# print(f'  Context Relevance    : {metrics["context_relevance"]:.2f}')
# print(f'  Context Utilization  : {metrics["context_utilization"]:.2f}')
# print(f'  Completeness         : {metrics["completeness"]:.2f}')
# print(f'  Adherence            : {metrics["adherence"]:.2f}')
# print(f'\nJudge reasoning: {eval_result.reasoning}')

print('Cell is commented out — uncomment when Ollama is running.')